In [0]:
from pyspark.sql.functions import *
from pyspark.sql import *

In [0]:
raw_fire_df = spark.read\
    .format("csv")\
    .options(header="true", inferSchema="true")\
        .load("/databricks-datasets/learning-spark-v2/sf-fire/sf-fire-calls.csv")

In [0]:
renamed_fire_df = raw_fire_df \
    .withColumnRenamed("Call Number", "CallNumber") \
    .withColumnRenamed("Unit ID", "UnitID") \
    .withColumnRenamed("Incident Number", "IncidentNumber") \
    .withColumnRenamed("Call Date", "CallDate") \
    .withColumnRenamed("Watch Date", "WatchDate") \
    .withColumnRenamed("Call Final Disposition", "CallFinalDisposition") \
    .withColumnRenamed("Available DtTm", "AvailableDtTm") \
    .withColumnRenamed("Zipcode of Incident", "Zipcode") \
    .withColumnRenamed("Station Area", "StationArea") \
    .withColumnRenamed("Final Priority", "FinalPriority") \
    .withColumnRenamed("ALS Unit", "ALSUnit") \
    .withColumnRenamed("Call Type Group", "CallTypeGroup") \
    .withColumnRenamed("Unit sequence in call dispatch", "UnitSequenceInCallDispatch") \
    .withColumnRenamed("Fire Prevention District", "FirePreventionDistrict") \
    .withColumnRenamed("Supervisor District", "SupervisorDistrict")
display(renamed_fire_df)

In [0]:
renamed_fire_df.printSchema()

In [0]:
#Change the datatype of AvailableDtTm
fire_df = renamed_fire_df\
    .withColumn("AvailableDtTm", to_timestamp("AvailableDtTm", "MM/dd/yyyy HH:mm:ss a"))\
        .withColumn("Delay",round("Delay",2))
display(fire_df)

In [0]:
fire_df.cache()

##### Q1. How many distinct types of calls were made to the Fire Department?
select count(distinct callType) as distinct_call_type_count
from demo_db.fire_service_calls_tbl
where callType is not null

In [0]:
#Spark SQL
fire_df.createOrReplaceTempView("fire_service_calls_view")
q1_sql_df = spark.sql("""select count(distinct callType) as distinct_call_type_count from fire_service_calls_view where callType is not null""")
display(q1_sql_df)

In [0]:
#q1_py_df = fire_df.select("callType").filter("callType is not null").distinct().count()
#display(q1_py_df)
q1_py_df = fire_df.where("callType is not null").select("callType").distinct()
print(q1_py_df.count())

##### Q2. What were distinct types of calls made to the Fire Department?
select distinct callType as distinct_call_types
from demo_db.fire_service_calls_tbl
where callType is not null

In [0]:
q2_py_df = fire_df.where("callType is not null").select(expr("callType as distinct_call_type")).distinct()
display(q2_py_df)

##### Q3. Find out all response for delayed times greater than 5 mins?
select callNumber, Delay
from demo_db.fire_service_calls_tbl
where Delay > 5

In [0]:
q3_df = fire_df.where("Delay > 5").select("callNumber", "Delay")
display(q3_df)

##### Q4. What were the most common call types?
select callType, count(*) as count
from demo_db.fire_service_calls_tbl
where callType is not null
group by callType
order by count desc

In [0]:
q4_df = fire_df\
    .where("callType is not null")\
    .select("callType")\
    .groupBy("callType")\
    .agg(count("callType").alias("count"))\
    .orderBy(desc("count"))
display(q4_df)

##### Q5. What zip codes accounted for most common calls?
select callType, zipCode, count(*) as count
from demo_db.fire_service_calls_tbl
where callType is not null
group by callType, zipCode
order by count desc


In [0]:
q5_df = fire_df\
    .where("callType is not null")\
    .groupBy("callType", "zipCode")\
    .agg(count("*").alias("count"))\
    .orderBy("callType", "zipCode")
display(q5_df)

##### Q6. What San Francisco neighborhoods are in the zip codes 94102 and 94103?
select zipCode, neighborhood
from demo_db.fire_service_calls_tbl
where zipCode == 94102 or zipCode == 94103

In [0]:
q6_df = fire_df\
    .where("zipCode IN (94102,94103)")\
        .select("zipCode", "neighborhood").distinct()
display(q6_df)

#####Q7. What was the sum of all call alarms, average, min, and max of the call response times?
select sum(NumAlarms), avg(Delay), min(Delay), max(Delay)
from demo_db.fire_service_calls_tbl

In [0]:
from pyspark.sql.functions import sum,max,avg,min
q7_df = fire_df.agg(sum("NumAlarms").alias("sum_of_num_alarms")\
    ,avg("Delay").alias("avg_of_delay")\
    ,max("Delay").alias("max_of_delay")\
    ,min("Delay").alias("min_of_delay"))
display(q7_df)

##### Q8. How many distinct years of data is in the data set?
select distinct year(to_date(callDate, "MM/dd/yyyy")) as year_num
from demo_db.fire_service_calls_tbl
order by year_num

In [0]:
q8_df = fire_df\
    .withColumn("year_num",year("callDate"))\
        .select("year_num")\
        .distinct()\
            .orderBy('year_num')

display(q8_df)

##### Q9. What week of the year in 2018 had the most fire calls?
select weekofyear(to_date(callDate, "MM/dd/yyyy")) week_year, count(*) as count
from demo_db.fire_service_calls_tbl
where year(to_date(callDate, "MM/dd/yyyy")) == 2018
group by week_year
order by count desc

In [0]:
q9_df = fire_df\
  .where("year(callDate) = 2018")\
    .select("calldate")\
      .withColumn("week_year", weekofyear("calldate"))\
        .groupBy("week_year")\
        .agg(count("week_year").alias("count"))\
        .orderBy(desc("count"))
display(q9_df)

####Q10. What neighborhoods in San Francisco had the worst response time in 2018?
select neighborhood, delay
from demo_db.fire_service_calls_tbl
where year(to_date(callDate, "MM/dd/yyyy")) == 2018
order by delay desc

In [0]:
q10_df = fire_df\
  .where("year(callDate) = 2018")\
    .select("neighborhood", "delay")\
      .orderBy(desc("delay"))
display(q10_df)